# <span style="color:#0F766E">Reading the Ground Before Modeling</span>

<table width="100%" cellpadding="10" cellspacing="0">
<tr bgcolor="#f8fafc"><td>
<p><b>Fundamentals of Natural Language Processing</b> | Universitat Autonoma de Barcelona | 2025 2026</p>
<p>Phoebe Iglesias (1713459), David Redrejo (1790336), Pau Rossell (1750424)</p>
<h3><font color="#0F766E">Notebook question</font></h3>
<p>What are these files really asking us to model?</p>
<h3><font color="#0F766E">Connection with the previous notebook</font></h3>
<p>This is where our story begins. We use it to decide whether the task is long clinical document coding or short literal normalization.</p>
<h3><font color="#0F766E">What this chapter contributes</font></h3>
<p>We read the data before trusting any model. Every later choice in the project is anchored here.</p>
</td></tr>
</table>

<table width="100%" cellpadding="8" cellspacing="0">
<tr bgcolor="#0F766E"><th><font color="white">Move</font></th><th><font color="white">What we try to understand</font></th></tr>
<tr><td>1</td><td>File roles</td></tr><tr><td>2</td><td>Overlap</td></tr><tr><td>3</td><td>Surface patterns</td></tr><tr><td>4</td><td>Ambiguity</td></tr><tr><td>5</td><td>ICD catalog</td></tr><tr><td>6</td><td>Code structure</td></tr><tr><td>7</td><td>Retrieval test</td></tr><tr><td>8</td><td>Modeling path</td></tr>
</table>

## Chapter Map

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

chapter_color = "#0F766E"
labels = ['File roles', 'Overlap', 'Surface patterns', 'Ambiguity', 'ICD catalog', 'Code structure', 'Retrieval test', 'Modeling path']

fig, ax = plt.subplots(figsize=(14, 2.6))
ax.set_xlim(0, len(labels))
ax.set_ylim(0, 1)
ax.axis("off")

for idx, label in enumerate(labels):
    card = FancyBboxPatch(
        (idx + 0.06, 0.25), 0.88, 0.48,
        boxstyle="round,pad=0.04,rounding_size=0.05",
        linewidth=1.4,
        edgecolor=chapter_color,
        facecolor="#f8fafc"
    )
    ax.add_patch(card)
    ax.text(idx + 0.5, 0.49, label, ha="center", va="center", fontsize=10.5, color="#0f172a", wrap=True)
    if idx < len(labels) - 1:
        ax.annotate("", xy=(idx + 1.02, 0.49), xytext=(idx + 0.94, 0.49), arrowprops=dict(arrowstyle=">", color=chapter_color, lw=1.8))

ax.text(0.02, 0.9, "How this notebook moves", fontsize=14, weight="bold", color=chapter_color)
plt.show()

## File Roles

We start as a group by asking what each file is allowed to tell us. This matters because mixing up training data, leaderboard data, and catalog data would make every later result look cleaner than it really is. The first table is our sanity check before any modeling happens.

In [ ]:
import warnings, re, unicodedata, math, random
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "data").exists() else NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"

required_files = [
    DATA_DIR / "leaderboard_data.csv",
    DATA_DIR / "codification_data.csv",
    DATA_DIR / "icd_d_p_pairs.csv",
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required data files. Place the project CSV files in the data/ folder: "
        + ", ".join(missing)
    )

leaderboard = pd.read_csv(DATA_DIR / "leaderboard_data.csv")
codification = pd.read_csv(DATA_DIR / "codification_data.csv")
icd_catalog = pd.read_csv(DATA_DIR / "icd_d_p_pairs.csv")

def normalize_text(text: str) -> str:
    text = str(text).strip().lower()
    text = "".join(
        ch for ch in unicodedata.normalize("NFD", text)
        if unicodedata.category(ch) != "Mn"
    )
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def add_surface_features(df, text_col):
    out = df.copy()
    s = out[text_col].astype(str)
    out["norm"] = s.map(normalize_text)
    out["n_chars"] = s.str.len()
    out["n_tokens"] = s.str.split().str.len()
    out["has_digit"] = s.str.contains(r"\d", regex=True)
    out["has_punct"] = s.str.contains(r"[^\w\s]", regex=True)
    out["is_all_upper"] = s.str.fullmatch(r"[^a-z???????]*[A-Z???????][^a-z???????]*", na=False)
    out["has_accent"] = s.map(lambda x: any("WITH" in unicodedata.name(ch, "") for ch in x if ch.isalpha()))
    return out

leaderboard = add_surface_features(leaderboard, "Literal")
codification = add_surface_features(codification, "Literal")
icd_catalog = add_surface_features(icd_catalog, "Description")

print(f"Project root : {PROJECT_ROOT}")
print(f"Data folder  : {DATA_DIR}")

display(leaderboard.head())
display(codification.head())
display(icd_catalog.head())

### File Roles Output

The output separates the project into three actors. The leaderboard file is what we must predict, the codification file is what we can learn from, and the ICD catalog is external knowledge. From here we know that the catalog can support us, but it cannot replace the supervised pairs.

In [ ]:

summary = pd.DataFrame([
    {
        "dataset": "leaderboard_data",
        "rows": len(leaderboard),
        "columns": ", ".join(leaderboard.columns[:2]),
        "unique_codes": np.nan,
        "unique_literals_or_desc": leaderboard["Literal"].nunique(),
    },
    {
        "dataset": "codification_data",
        "rows": len(codification),
        "columns": ", ".join(codification.columns[:2]),
        "unique_codes": codification["Code"].nunique(),
        "unique_literals_or_desc": codification["Literal"].nunique(),
    },
    {
        "dataset": "icd_d_p_pairs",
        "rows": len(icd_catalog),
        "columns": ", ".join(icd_catalog.columns[:3]),
        "unique_codes": icd_catalog["Code"].nunique(),
        "unique_literals_or_desc": icd_catalog["Description"].nunique(),
    },
])
display(summary)


## Overlap After Normalization

Once the file roles are clear, we ask whether the task is partly a memory problem. We compare exact literals and normalized literals because medical text often changes through accents, casing, punctuation, and tiny formatting differences. If normalization increases overlap, then a lexical model has a real reason to work.

In [ ]:

leaderboard_exact_in_train = leaderboard["Literal"].isin(set(codification["Literal"])).mean()
leaderboard_norm_in_train = leaderboard["norm"].isin(set(codification["norm"])).mean()

cod_with_catalog = codification["Code"].isin(set(icd_catalog["Code"])).mean()
exact_train_literal_in_catalog_desc = codification["Literal"].isin(set(icd_catalog["Description"])).mean()
norm_train_literal_in_catalog_desc = codification["norm"].isin(set(icd_catalog["norm"])).mean()

connection = pd.DataFrame({
    "check": [
        "Leaderboard literal exactly seen in training literals",
        "Leaderboard literal seen after normalization",
        "Training code present in ICD catalog",
        "Training literal exactly matches ICD official description",
        "Training literal matches ICD description after normalization",
    ],
    "share": [
        leaderboard_exact_in_train,
        leaderboard_norm_in_train,
        cod_with_catalog,
        exact_train_literal_in_catalog_desc,
        norm_train_literal_in_catalog_desc,
    ]
})
display(connection.style.format({"share": "{:.1%}"}))


## Literal Surface Patterns

The overlap check tells us that text form matters, so we inspect the form itself. We measure length, token count, digits, punctuation, uppercase text, and accents. We are trying to see whether the literals behave like full sentences or like compact clinical labels.

In [ ]:

surface_stats = pd.DataFrame([
    {
        "dataset": "leaderboard literals",
        "mean chars": leaderboard["n_chars"].mean(),
        "mean tokens": leaderboard["n_tokens"].mean(),
        "contains digit": leaderboard["has_digit"].mean(),
        "contains punctuation": leaderboard["has_punct"].mean(),
        "all uppercase": leaderboard["is_all_upper"].mean(),
        "contains accent": leaderboard["has_accent"].mean(),
    },
    {
        "dataset": "training literals",
        "mean chars": codification["n_chars"].mean(),
        "mean tokens": codification["n_tokens"].mean(),
        "contains digit": codification["has_digit"].mean(),
        "contains punctuation": codification["has_punct"].mean(),
        "all uppercase": codification["is_all_upper"].mean(),
        "contains accent": codification["has_accent"].mean(),
    },
    {
        "dataset": "ICD official descriptions",
        "mean chars": icd_catalog["n_chars"].mean(),
        "mean tokens": icd_catalog["n_tokens"].mean(),
        "contains digit": icd_catalog["has_digit"].mean(),
        "contains punctuation": icd_catalog["has_punct"].mean(),
        "all uppercase": icd_catalog["is_all_upper"].mean(),
        "contains accent": icd_catalog["has_accent"].mean(),
    },
])

display(surface_stats.style.format({
    "mean chars": "{:.2f}",
    "mean tokens": "{:.2f}",
    "contains digit": "{:.1%}",
    "contains punctuation": "{:.1%}",
    "all uppercase": "{:.1%}",
    "contains accent": "{:.1%}",
}))


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(leaderboard["n_chars"], bins=30, alpha=0.6, label="leaderboard")
axes[0].hist(codification["n_chars"], bins=30, alpha=0.6, label="training")
axes[0].set_title("Literal length in characters")
axes[0].legend()

token_bins = np.arange(1, 12) - 0.5
axes[1].hist(leaderboard["n_tokens"], bins=token_bins, alpha=0.6, label="leaderboard")
axes[1].hist(codification["n_tokens"], bins=token_bins, alpha=0.6, label="training")
axes[1].set_title("Literal length in tokens")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:

leader_vals = [
    leaderboard["has_digit"].mean(),
    leaderboard["has_punct"].mean(),
    leaderboard["is_all_upper"].mean(),
    leaderboard["has_accent"].mean(),
]
train_vals = [
    codification["has_digit"].mean(),
    codification["has_punct"].mean(),
    codification["is_all_upper"].mean(),
    codification["has_accent"].mean(),
]
features = ["digit", "punctuation", "all uppercase", "accent"]
x = np.arange(len(features))
width = 0.35

plt.figure(figsize=(10,5))
plt.bar(x - width/2, leader_vals, width, label='leaderboard')
plt.bar(x + width/2, train_vals, width, label='training')
plt.xticks(x, features)
plt.ylim(0, 0.35)
plt.ylabel('share')
plt.title('Surface phenomena that matter for preprocessing')
plt.legend()
plt.show()


### Surface Pattern Reading

The plots make the task feel much more concrete. These are short pieces of language, often closer to labels than to clinical notes. That is why character fragments become important, and why the next notebooks do not start with a large model first.

## Ambiguity and Synonymy

After looking at the surface, we check whether the labels themselves are clean. A literal can point to several codes, and one code can be written in many ways. This is the first place where we see that a lookup table would be too fragile.

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

literal_to_n_codes.value_counts().sort_index().head(10).plot(kind="bar", ax=axes[0])
axes[0].set_title("How many codes can one literal map to?")
axes[0].set_xlabel("# codes for one literal")
axes[0].set_ylabel("# literals")

code_to_n_literals.value_counts().sort_index().head(15).plot(kind="bar", ax=axes[1])
axes[1].set_title("How many literals can describe one code?")
axes[1].set_xlabel("# literals for one code")
axes[1].set_ylabel("# codes")

plt.tight_layout()
plt.show()


In [ ]:

ambiguous_examples = (
    codification.groupby("Literal")["Code"].nunique()
    .sort_values(ascending=False)
    .head(8)
    .index
)
display(
    codification[codification["Literal"].isin(ambiguous_examples)]
    .sort_values(["Literal", "Code"])
    .head(40)
)


### From Ambiguity to a Trainable Target

The ambiguity does not disappear, but we need a target that a classifier can learn. We therefore keep the most frequent category for each repeated literal. This is a practical compromise, and we carry that limitation into the evaluation instead of pretending it is gone.

## ICD Catalog Signal

Now that we understand the supervised data, we test the catalog. The question is whether official ICD descriptions use the same words as the real clinical literals. We use token overlap as a simple but honest diagnostic.

In [ ]:

merged = codification.merge(
    icd_catalog[["Code", "D_P", "Description", "norm"]],
    on="Code",
    how="left",
    suffixes=("_literal", "_description")
)

known_catalog = merged.dropna(subset=["Description"]).copy()

def token_jaccard(a, b):
    sa, sb = set(str(a).split()), set(str(b).split())
    return len(sa & sb) / len(sa | sb) if (sa or sb) else np.nan

known_catalog["token_jaccard"] = [
    token_jaccard(a, b)
    for a, b in zip(known_catalog["norm_literal"], known_catalog["norm_description"])
]

display(
    known_catalog[["Code", "Literal", "Description", "D_P", "token_jaccard"]]
    .sort_values("token_jaccard", ascending=False)
    .head(10)
)


In [ ]:

plt.figure(figsize=(10, 5))
plt.hist(known_catalog["token_jaccard"].dropna(), bins=20)
plt.title("Lexical overlap between training literal and official ICD description")
plt.xlabel("token Jaccard overlap")
plt.ylabel("count")
plt.show()


### Catalog Overlap Reading

The overlap is useful mainly because it tells us what not to do. Official descriptions are longer and more formal than the literals. The catalog is knowledge, not a direct dictionary.

## Code Structure

The text is only half of the problem. ICD codes themselves have structure, so we inspect diagnosis and procedure patterns, code length, and broad families. This explains why predicting the first character is a meaningful first target.

In [ ]:

catalog_dp = icd_catalog["D_P"].value_counts().rename_axis("type").reset_index(name="count")
display(catalog_dp)

plt.figure(figsize=(7, 4))
plt.bar(catalog_dp['type'], catalog_dp['count'])
plt.title('Diagnosis vs procedure codes in ICD catalog')
plt.xlabel('type')
plt.ylabel('count')
plt.show()


In [ ]:

merged["code_len"] = merged["Code"].astype(str).str.len()
merged["code_type"] = merged["Code"].astype(str).str.match(r"^[A-Z]").map({True: "alphanumeric", False: "numeric"})

code_structure = (
    merged.groupby(["code_type", "D_P"])["Code"]
    .count()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(code_structure)
ct = merged.pivot_table(index='code_len', columns='D_P', values='Code', aggfunc='count', fill_value=0)
ct.plot(kind='bar', figsize=(9,5))
plt.title('Length of training codes, split by diagnosis/procedure when available')
plt.xlabel('code length')
plt.ylabel('count')
plt.show()


### Code Structure Reading

The structure confirms that categories are uneven. Some categories have many examples and others are rare. This will later explain why strict accuracy, macro F1, and class weighting can disagree.

## Retrieval Baseline

At this point we have enough evidence to try a deliberately simple model. We retrieve the closest training literal with TF IDF and cosine similarity. If this works at all, then the lexical surface is carrying useful signal.

In [ ]:

def retrieval_baseline(df, random_state=42, strict_by_literal=True, top_k=5):
    work = df.copy()
    work["norm_literal"] = work["Literal"].map(normalize_text)

    if strict_by_literal:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)
        train_idx, test_idx = next(splitter.split(work, groups=work["norm_literal"]))
        setting = "strict split by normalized literal"
    else:
        train_idx, test_idx = train_test_split(
            np.arange(len(work)), test_size=0.2, random_state=random_state
        )
        setting = "random split"

    train = work.iloc[train_idx].copy()
    test = work.iloc[test_idx].copy()

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
    X_train = vectorizer.fit_transform(train["norm_literal"])
    X_test = vectorizer.transform(test["norm_literal"])

    sims = linear_kernel(X_test, X_train)
    top_idx = np.argpartition(-sims, kth=top_k-1, axis=1)[:, :top_k]
    row_idx = np.arange(sims.shape[0])[:, None]
    top_idx = top_idx[row_idx, np.argsort(-sims[row_idx, top_idx], axis=1)]

    train_codes = train["Code"].to_numpy()
    pred_top1 = train_codes[top_idx[:, 0]]
    pred_topk = train_codes[top_idx]

    return {
        "setting": setting,
        "n_train": len(train),
        "n_test": len(test),
        "acc@1": (pred_top1 == test["Code"].to_numpy()).mean(),
        "hit@5": np.any(pred_topk == test["Code"].to_numpy()[:, None], axis=1).mean(),
        "examples": pd.DataFrame({
            "literal": test["Literal"].values[:10],
            "gold_code": test["Code"].values[:10],
            "pred_code_top1": pred_top1[:10],
        })
    }

strict_results = retrieval_baseline(codification, strict_by_literal=True)
random_results = retrieval_baseline(codification, strict_by_literal=False)

baseline_table = pd.DataFrame([
    {k: v for k, v in strict_results.items() if k != "examples"},
    {k: v for k, v in random_results.items() if k != "examples"},
])

display(baseline_table.style.format({"acc@1": "{:.1%}", "hit@5": "{:.1%}"}))


### Random Split and Strict Literal Split

We evaluate in two ways because we do not want to fool ourselves. A random split can reward repeated wording. A strict split by normalized literal asks whether the method survives when the wording is genuinely new.

In [ ]:

display(strict_results["examples"])


## First Baseline Reading

The retrieval result becomes our compass. It shows that lexical similarity is useful, but also that ambiguity and rare formulations need a classifier with more flexibility.

## Modeling Path

The next move is <b><font color="#2563EB">Notebook 02: Building the First Classical Baseline</font></b>. We do not jump to a transformer yet. We first build a strong classical model because the EDA has shown that small fragments and normalization matter.

## EDA Takeaways

We close this notebook with a clearer task definition. We are coding short clinical literals, not long documents. That single understanding shapes the rest of the project.